# Bonus 2: fine-tuning an open "System One" decision model (Laya)

**Outside the course scope; an easter egg only.** [Laya](https://huggingface.co/convaiinnovations/laya) (Apache-2.0, ModernBERT-large, 421M parameters) is an open model with the same interface as TypeSafe's hosted Jev. You give it a **state** (text or JSON) and **typed questions** (choice, yes/no, score), and it returns calibrated probabilities instead of generated text. We use it in two ways:

1. **Zero-shot mood.** On 14,640 **human-labeled** airline tweets, does it read mood better than our lexicon-trained classifier (macro-F1 0.49 in the main notebook, section 3)?
2. **Fine-tuned reach.** We feed each tweet **plus its author card** (followers, Klout, country, language, weekday, hour) as one JSON state and ask "will it be retweeted?". We fine-tune on the main notebook's training users and test on its held-out users, then compare with zero-shot Laya, fine-tuned DistilBERT and LogisticRegression.

No API key is needed. The workflow has three steps:

1. **This notebook** writes the inputs to `data/laya_job/` in seconds.
2. **`laya_gpu_job.py`** runs zero-shot scoring and two fine-tunes. It takes about 1 hour and can run on any free GPU; we used an RTX 4060 laptop.
3. **This notebook** reads the predictions back and computes the metrics.

Extra packages: `torch`, `laya`. Run `final_analysis.ipynb` first.

In [ ]:
import json
import logging
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(message)s", force=True)
log = logging.getLogger("moodreach.laya")
sns.set_theme(style="whitegrid")
SEED = 42
JOB = "data/laya_job"  # inputs and outputs of laya_gpu_job.py (git-ignored)
os.makedirs(JOB, exist_ok=True)
main = json.load(open("results.json"))


def save(name: str) -> None:
    plt.savefig(f"figures/{name}.png", dpi=120, bbox_inches="tight")
    log.info("saved figures/%s.png", name)

## 1. Build the job inputs

We use the same cleaning, ≤ 20 tweets per user and user-grouped 80/20 split as the main notebook. Each test tweet gets two states:

- **text:** the tweet alone, for a fair comparison with DistilBERT.
- **card:** the tweet plus an author card (followers, Klout, gender, country, language, weekday, hour), so one model reads both.

In [ ]:
SENTIMENT_Q = {
    "sentiment": {
        "type": "choice",
        "instructions": "What overall sentiment does the author express in this tweet?",
        "criteria": {
            "negative": "complaint, frustration, anger, disappointment or sarcasm",
            "neutral": "factual statement, question, announcement or no clear emotion",
            "positive": "praise, thanks, excitement or happiness",
        },
    }
}
CONTEXT = "This is an original tweet from early 2016 (mostly tech and cloud-computing Twitter)"
QUESTIONS = {
    "sentiment": SENTIMENT_Q,
    "reach_text": {"retweeted": {"type": "noul", "instructions": f"{CONTEXT}. Will at least one other user retweet it?"}},
    "reach_card": {"retweeted": {"type": "noul", "instructions": f"{CONTEXT} with a card about its author. Will at least one other user retweet it?"}},
}

path = hf_hub_download("haoxianc/paakhim10_tweets-and-engagement-metrics", "tweets-engagement-metrics.csv", repo_type="dataset", local_dir="data")
df = pd.read_csv(path, low_memory=False).drop(columns=["Unnamed: 0"]).drop_duplicates("TweetID").reset_index(drop=True)
df["text"] = df["text"].str.strip().str.strip('"').str.strip()
orig = df[~df.IsReshare].reset_index(drop=True)
rank = orig.assign(_r=np.random.default_rng(SEED).random(len(orig))).groupby("UserID")._r.rank(method="first")
capped = orig[rank <= 20].reset_index(drop=True)
assert len(capped) == main["data"]["capped_tweets"], "cleaning differs from final_analysis.ipynb"
y = (capped.RetweetCount > 0).to_numpy()
tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(capped, y, groups=capped.UserID))


def card(r) -> dict:
    return {
        "tweet": r.text,
        "author_followers": int(r.Reach),
        "author_klout_score": int(r.Klout),
        "author_gender": r.Gender,
        "country": r.Country,
        "language": r.Lang,
        "weekday": r.Weekday,
        "hour_us_mountain": int(r.Hour),
    }


json.dump(QUESTIONS, open(f"{JOB}/questions.json", "w"), indent=2)
for variant, make in [("text", lambda r: {"tweet": r.text}), ("card", card)]:
    with open(f"{JOB}/train_{variant}.jsonl", "w") as f:
        for r in capped.iloc[tr_idx].itertuples():
            f.write(json.dumps({"state": make(r), "questions": QUESTIONS[f"reach_{variant}"], "expected": {"retweeted": bool(r.RetweetCount > 0)}}) + "\n")
test = capped.iloc[te_idx]
pd.DataFrame(
    {
        "TweetID": test.TweetID.to_numpy(),
        "retweeted": y[te_idx].astype(int),
        "state_text": [json.dumps({"tweet": r.text}) for r in test.itertuples()],
        "state_card": [json.dumps(card(r)) for r in test.itertuples()],
    }
).to_csv(f"{JOB}/test.csv", index=False)
air = pd.read_csv(hf_hub_download("osanseviero/twitter-airline-sentiment", "Tweets.csv", repo_type="dataset", local_dir="data"))
air[["tweet_id", "airline_sentiment", "text"]].to_csv(f"{JOB}/airline.csv", index=False)
log.info("job inputs in %s: train %d, test %d, airline %d", JOB, len(tr_idx), len(te_idx), len(air))
log.info("example card: %s", json.dumps(card(next(test.itertuples())))[:300])

## 2. Run the GPU job (outside this notebook)

```bash
python laya_gpu_job.py --work data/laya_job            # same machine, or:
scp -r data/laya_job gpu-box:moodreach-laya/ && ssh gpu-box python laya_gpu_job.py --work moodreach-laya/laya_job
```

Then copy `pred_*.csv` and `meta_*.json` back into `data/laya_job/`. On an 8 GB GPU, `--micro-batch 4` with AMP and gradient checkpointing fits ModernBERT-large.

In [ ]:
needed = ["pred_airline.csv", "pred_zero_shot.csv", "pred_finetuned_text.csv", "pred_finetuned_card.csv"]
missing = [n for n in needed if not os.path.exists(f"{JOB}/{n}")]
assert not missing, f"run laya_gpu_job.py first; missing {missing}"
meta = {v: json.load(open(f"{JOB}/meta_{v}.json")) for v in ["text", "card"] if os.path.exists(f"{JOB}/meta_{v}.json")}
log.info("job meta: %s", meta)

## 3. Zero-shot mood on human-labeled airline tweets

In [ ]:
pa = pd.read_csv(f"{JOB}/pred_airline.csv")
lex = {r["model"]: r for r in main["classifier"]["airline_transfer"]}
best = main["classifier"]["best"]
mood_table = pd.DataFrame(
    {
        "Majority baseline (always 'negative')": lex["Majority baseline (always 'negative')"],
        "Count + MultinomialNB (lexicon labels)": lex["Count + MultinomialNB"],
        f"{best} (lexicon labels)": lex[best],
        "Laya zero-shot (no training)": {"accuracy": accuracy_score(pa.airline_sentiment, pa.laya_zero_shot), "macro_F1": f1_score(pa.airline_sentiment, pa.laya_zero_shot, average="macro")},
    }
).T[["accuracy", "macro_F1"]].astype(float).round(3)
log.info("human-labeled airline tweets:\n%s", mood_table.to_string())

order = ["negative", "neutral", "positive"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].barh(mood_table.index[::-1], mood_table.macro_F1[::-1], color=["#d1495b", "#3b6fb6", "#3b6fb6", "#bbbbbb"])
for i, v in enumerate(mood_table.macro_F1[::-1]):
    axes[0].text(v + 0.01, i, f"{v:.2f}", va="center")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("macro-F1 on 14,640 human-labeled airline tweets")
axes[0].set_title("Reading human mood: lexicon-trained vs zero-shot Laya")
ConfusionMatrixDisplay.from_predictions(pa.airline_sentiment, pa.laya_zero_shot, labels=order, normalize="true", values_format=".2f", cmap="Reds", ax=axes[1], colorbar=False)
axes[1].set_title("Laya zero-shot (row-normalized)")
plt.tight_layout()
save("7_2_laya_mood")
plt.show()
mood_table

## 4. Reach on held-out users: zero-shot vs fine-tuned

In [ ]:
pr = pd.read_csv(f"{JOB}/pred_zero_shot.csv")
for v in ["text", "card"]:
    pr = pr.merge(pd.read_csv(f"{JOB}/pred_finetuned_{v}.csv")[["TweetID", f"p_finetuned_{v}"]], on="TweetID")
assert len(pr) == len(te_idx)
bonus = json.load(open("bonus_results.json"))["test_auc"] if os.path.exists("bonus_results.json") else {}
main_auc = {r["model"]: r["ROC_AUC"] for r in main["logistic"]["table"]}
auc = pd.Series(
    {
        "Mood features only (LogReg)": main_auc["Mood features only"],
        "Laya zero-shot, text": roc_auc_score(pr.retweeted, pr.p_zero_text),
        "Laya zero-shot, text + author card": roc_auc_score(pr.retweeted, pr.p_zero_card),
        "All metadata, no text (LogReg)": main_auc["All features"],
        **({"Fine-tuned DistilBERT, text": bonus["Text: fine-tuned DistilBERT"]} if bonus else {}),
        "Fine-tuned Laya, text": roc_auc_score(pr.retweeted, pr.p_finetuned_text),
        "Fine-tuned Laya, text + author card": roc_auc_score(pr.retweeted, pr.p_finetuned_card),
    }
).round(4)
log.info("test ROC AUC, retweeted yes/no (held-out users):\n%s", auc.to_string())

fig, ax = plt.subplots(figsize=(9, 4.2))
palette = {"Mood": "#2e9e6a", "Laya zero": "#bbbbbb", "All metadata": "#7a4fb0", "Fine-tuned DistilBERT": "#3b6fb6", "Fine-tuned Laya": "#d1495b"}
colors = [next(c for k, c in palette.items() if name.startswith(k)) for name in auc.index]
ax.barh(auc.index[::-1], auc.values[::-1], color=colors[::-1])
ax.axvline(0.5, color="k", ls="--", lw=1)
for i, v in enumerate(auc.values[::-1]):
    ax.text(v + 0.004, i, f"{v:.3f}", va="center")
ax.set_xlim(0.45, max(0.85, auc.max() + 0.04))
ax.set_title("Will an original tweet be retweeted? Test ROC AUC, unseen users")
plt.tight_layout()
save("7_3_laya_reach")
plt.show()

json.dump(
    {
        "base": "convaiinnovations/laya",
        "train_rows": int(len(tr_idx)),
        "test_rows": int(len(te_idx)),
        "jobs": meta,
        "airline_mood": mood_table.reset_index().rename(columns={"index": "model"}).to_dict("records"),
        "reach_auc": auc.to_dict(),
    },
    open("laya_results.json", "w"),
    indent=2,
    default=str,
)
auc

**Reading it:**

- **Mood is a language task.** A pretrained decision model reads human mood zero-shot better than a classifier trained on lexicon labels.
- **Reach depends on context.** Zero-shot, the model cannot know what spread on 2016 tech Twitter. Fine-tuning teaches it, and the author card shows how much the author adds on top of the words.

The headline of the main notebook stays the same: author signals carry most of the information and mood adds little. These models are not interpretable, so they stay a bonus.